In [2]:
import os; os.environ['CUDA_VISIBLE_DEVICES'] = '1'

In [1]:
import torch; torch.cuda.empty_cache()
!nvidia-smi

Mon Apr  6 16:21:48 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.90.07              Driver Version: 550.90.07      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA H100 80GB HBM3          On  |   00000000:A4:00.0 Off |                    0 |
| N/A   28C    P0            109W /  700W |   41420MiB /  81559MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
import json

test_data = []
with open('data/test.jsonl', 'r') as f:
    for line in f.readlines():
        if line.strip():
            data = json.loads(line)
            test_data.append(data)

print(test_data[0])

{'expression': '55 * 88 * 59 - 71', 'answer': 285489, 'type': 'standard'}


In [3]:
from datasets import Dataset
dataset = Dataset.from_list(test_data)
print(dataset)

Dataset({
    features: ['expression', 'answer', 'type'],
    num_rows: 400
})


In [4]:
SYSTEM_PROMPT = (
    "Evaluate the arithmetic expression using the symbol definitions provided. "
    "There are 4 symbols θ, α, γ, β each representing one of the four basic arithmetic operations (+, -, ×, ÷). "
    "Each symbol maps to exactly one operation. Standard operator precedence (BODMAS) applies. "
    "Be concise. Put your final answer in \\boxed{}."
)

SYSTEM_PROMPT

'Evaluate the arithmetic expression using the symbol definitions provided. There are 4 symbols θ, α, γ, β each representing one of the four basic arithmetic operations (+, -, ×, ÷). Each symbol maps to exactly one operation. Standard operator precedence (BODMAS) applies. Be concise. Put your final answer in \\boxed{}.'

In [5]:
dataset = dataset.map(lambda x: {
    "prompt": [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Expression: {x['expression']} ->/no_think"}
    ]
})
dataset[0]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

{'expression': '55 * 88 * 59 - 71',
 'answer': 285489,
 'type': 'standard',
 'prompt': [{'content': 'Evaluate the arithmetic expression using the symbol definitions provided. There are 4 symbols θ, α, γ, β each representing one of the four basic arithmetic operations (+, -, ×, ÷). Each symbol maps to exactly one operation. Standard operator precedence (BODMAS) applies. Be concise. Put your final answer in \\boxed{}.',
   'role': 'system'},
  {'content': 'Expression: 55 * 88 * 59 - 71 ->/no_think', 'role': 'user'}]}

In [6]:
import re
ptrn = re.compile(r"\\boxed\{(.*)\}")

a = ptrn.search("1 + 2 = \\boxed{3}")
print(a.group(1))

b = ptrn.search("1 + 2 = 3")
print(b)

3
None


In [7]:
def correctness_reward_fn(prompts, completions, answer, **kwargs) -> list[float]:
    responses = [completion[0]["content"] for completion in completions]
    # extract the boxed answer from each response
    rewards = []
    for response, ans in zip(responses, answer):
        try:
            _ans = ptrn.search(response)
            rewards.append(float(float(_ans.group(1)) == float(ans)))
        except Exception as e:
            rewards.append(0.0)
        # print(f"response: {response}, true_ans: {ans}, reward: {rewards[-1]}")
    return rewards
    



In [8]:
# test correctness_reward_fn
prompts = ["1 + 2 = \\boxed{3}", "1 + 2 = 3"]
completions = [[{'content':"1 + 2 = \\boxed{3}"}], [{'content':"1 + 2 = 4"}]]
answer = ["3", "4"]
correctness_reward_fn(prompts, completions, answer)

[1.0, 0.0]

In [9]:
from unsloth import FastLanguageModel

DEFAULT_MODEL_NAME = "outputs/grpo-api-adapter-cold-start-training-0/checkpoint-3000"
DEFAULT_MAX_SEQ_LENGTH = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=DEFAULT_MODEL_NAME,
    max_seq_length=DEFAULT_MAX_SEQ_LENGTH,
    dtype=None,  # auto-detect (bf16 on H100)
    load_in_4bit=False,
    fast_inference=True,
    gpu_memory_utilization=0.5,
)
FastLanguageModel.for_inference(model)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
INFO 04-06 16:22:43 [vllm_utils.py:724] Unsloth: Patching vLLM v1 graph capture
==((====))==  Unsloth 2026.3.4: Fast Qwen3 patching. Transformers: 4.57.6. vLLM: 0.17.0.
   \\   /|    NVIDIA H100 80GB HBM3. Num GPUs = 8. Max memory: 79.097 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 9.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: vLLM loading unsloth/Qwen3-8B with actual GPU utilization = 23.94%
Unsloth: Your GPU has CUDA compute capability 9.0 with VRAM = 79.1 GB.
Unsloth: Using conservativeness = 1.0. Chunked prefill tokens = 2048. Num Sequences = 16.
Unsloth: vLLM's KV Cache can use up to 3.21 GB. Also swap space = 6 

/workspace/home/lab/rawhad/api-adapter/.venv/lib/python3.10/site-packages/pydantic/type_adapter.py:605: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `enum` - serialized value may not be as expected [field_name='mode', input_value=3, input_type=int])
  return self.serializer.to_python(


INFO 04-06 16:22:56 [model.py:531] Resolved architecture: Qwen3ForCausalLM
INFO 04-06 16:22:56 [model.py:1554] Using max model len 2048
INFO 04-06 16:22:56 [scheduler.py:231] Chunked prefill is enabled with max_num_batched_tokens=2048.
INFO 04-06 16:22:56 [vllm.py:747] Asynchronous scheduling is enabled.
INFO 04-06 16:22:56 [core.py:101] Initializing a V1 LLM engine (v0.17.0) with config: model='unsloth/Qwen3-8B', speculative_config=None, tokenizer='unsloth/Qwen3-8B', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=2048, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_fallback=False, disable_any_whitespace=Fa

/workspace/home/lab/rawhad/api-adapter/.venv/lib/python3.10/site-packages/pydantic/type_adapter.py:605: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `enum` - serialized value may not be as expected [field_name='mode', input_value=3, input_type=int])
  return self.serializer.to_python(


[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
INFO 04-06 16:23:01 [parallel_state.py:1715] rank 0 in world size 1 is assigned as DP rank 0, PP rank 0, PCP rank 0, TP rank 0, EP rank N/A, EPLB rank N/A
INFO 04-06 16:23:02 [topk_topp_sampler.py:51] Using FlashInfer for top-p & top-k sampling.
INFO 04-06 16:23:02 [base.py:106] Offloader set to NoopOffloader
INFO 04-06 16:23:02 [gpu_model_runner.py:4255] Starting to load model unsloth/Qwen3-8B...
INFO 04-06 16:23:02 [cuda.py:405] Using FLASH_ATTN at

...

INFO 04-06 16:23:28 [core.py:282] init engine (profile, create kv cache, warmup model) took 21.05 seconds
INFO 04-06 16:23:29 [llm.py:388] Supported tasks: ('generate',)
Unsloth: Just some info: will skip parsing ['attention_norm', 'post_feedforward_layernorm', 'input_layernorm', 'post_attention_layernorm', 'ffn_norm', 'q_norm', 'norm2', 'norm1', 'norm', 'pre_feedforward_layernorm', 'k_norm', 'layer_norm1', 'layer_norm2', 'post_layernorm']


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Performing substitution for additional_keys=set()
Unsloth: Just some info: will skip parsing ['attention_norm', 'post_feedforward_layernorm', 'input_layernorm', 'post_attention_layernorm', 'ffn_norm', 'q_norm', 'norm2', 'cross_attn_input_layernorm', 'norm1', 'norm', 'pre_feedforward_layernorm', 'k_norm', 'layer_norm1', 'cross_attn_post_attention_layernorm', 'layer_norm2', 'post_layernorm']


Unsloth 2026.3.4 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen3ForCausalLM(
      (model): Qwen3Model(
        (embed_tokens): Embedding(151936, 4096, padding_idx=151654)
        (layers): ModuleList(
          (0-35): 36 x Qwen3DecoderLayer(
            (self_attn): Qwen3Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=32, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=32, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear

In [10]:
texts = [
    tokenizer.apply_chat_template(x['prompt'], tokenize=False, add_generation_prompt=True, enable_thinking=False)
    for x in dataset
]

from vllm import SamplingParams
sampling_params = SamplingParams(
    temperature=1.0,
    max_tokens=1024,
)

In [11]:
outputs = model.fast_generate(
    texts,
    sampling_params=sampling_params,
    lora_request=model.load_lora(DEFAULT_MODEL_NAME),
)

all_outputs = [o.outputs[0].text for o in outputs]
print(f"Inference complete.")




Rendering prompts:   0%|          | 0/400 [00:00<?, ?it/s]

WARNING 04-06 16:23:31 [input_processor.py:168] vLLM has deprecated support for supporting different tokenizers for different LoRAs. By default, vLLM uses base model's tokenizer. If you are using a LoRA with its own tokenizer, consider specifying `--tokenizer [lora_path]` to use the LoRA tokenizer.


Processed prompts:   0%|                                                              | 0/400 [00:00<?, ?it/s,…

Inference complete.


In [12]:
all_outputs[0]

"To evaluate the expression:\n\n**Expression:** `55 * 88 * 59 - 71`\n\nWe are told each symbol maps to an operation:\n\n- `θ` → `+`\n- `α` → `-`\n- `γ` → `×`\n- `β` → `÷`\n\nSo in this case, the expression uses only `×` (`γ`) and `-` (`α`) — no `θ` or `β`.\n\n---\n\nLet's apply the **BODMAS** rule:\n\n### Step 1: Perform all **multiplications** first.\n\nSo, `55 × 88 × 59`\n\nLet’s compute step-by-step:\n\n- `55 × 88 = 4840`\n- `4840 × 59`\n\nNow calculate `4840 × 59`:\n\nWe can break it:\n\n```\n4840 × 59 = 4840 × (60 - 1) = (4840 × 60) - 4840\n\n4840 × 60 = 290400  \n290400 - 4840 = 285560\n```\n\nSo, `55 × 88 × 59 = 285,560`\n\n### Step 2: Subtract 71 from the result:\n\n```\n285560 - 71 = 285489\n```\n\n---\n\n### ✅ Final Answer:\n\n$$\n\\boxed{285489}\n$$"

In [13]:
ground_truths = [x['answer'] for x in dataset]
# convert all_outputs to completions: list[list[dict]]
completions = [
    [{"content": o}]
    for o in all_outputs
]
rewards = correctness_reward_fn(texts, completions, ground_truths)
accuracy = sum(rewards) / len(rewards)
print(f"Accuracy: {accuracy}")

total_correct = sum(rewards)
total_incorrect = len(rewards) - total_correct

# standard correct
total_standard = len([1 for x in dataset if x['type'] == 'standard'])
standard_correct = len([1 for x, r in zip(dataset, rewards) if r == 1 and x['type'] == 'standard'])
standard_incorrect = total_standard - standard_correct

# custom correct
total_custom = len([1 for x in dataset if x['type'] == 'custom'])
custom_correct = len([1 for x, r in zip(dataset, rewards) if r == 1 and x['type'] == 'custom'])
custom_incorrect = total_custom - custom_correct

print(f"Overall:  {total_correct}/{len(dataset)} = {total_correct/len(dataset)*100}%")
print(f"Custom:   {custom_correct}/{len([x for x in dataset if x['type'] == 'custom'])} = {custom_correct/len([x for x in dataset if x['type'] == 'custom'])*100}%")
print(f"Standard: {standard_correct}/{len([x for x in dataset if x['type'] == 'standard'])} = {standard_correct/len([x for x in dataset if x['type'] == 'standard'])*100}%")





Accuracy: 0.9825
Overall:  393.0/400 = 98.25%
Custom:   196/200 = 98.0%
Standard: 197/200 = 98.5%


In [17]:
# see predictions for custom type
for i, (x, o) in enumerate(zip(dataset, all_outputs)):
    if x['type'] == 'custom':
        print(f"Prompt: {texts[i]}")
        print(f"Expression: {x['expression']}")
        print(f"Predicted: {o}")
        print(f"True: {x['answer']}")
        print("-"*100)


Prompt: <|im_start|>system
Evaluate the arithmetic expression using the symbol definitions provided. There are 4 symbols θ, α, γ, β each representing one of the four basic arithmetic operations (+, -, ×, ÷). Each symbol maps to exactly one operation. Standard operator precedence (BODMAS) applies. Be concise. Put your final answer in \boxed{}.<|im_end|>
<|im_start|>user
Expression: 36 γ 52 ->/no_think<|im_end|>
<|im_start|>assistant
<think>

</think>


Expression: 36 γ 52
Predicted: We are given a mapping of symbols to arithmetic operations. The mapping is:

- θ → addition (+)  
- α → subtraction (-)  
- γ → multiplication (×)  
- β → division (÷)  

Given expression:  
**36 γ 52**

Since γ maps to multiplication, this becomes:

**36 × 52**

Let's compute that using standard arithmetic:

$$
36 \times 52 = (36 \times 50) + (36 \times 2) = 1800 + 72 = 1872
$$

### Final Answer:
$$
\boxed{1872}
$$
True: 1872
----------------------------------------------------------------------------------